## AI Powered API Endpoint Analysis

### Requirements

```txt
pydantic
prance[osv]
agno
openapi
google-genai
ipywidgets
```

#### Imports

In [15]:
from pathlib import Path
import ipywidgets as widgets
from prance import ResolvingParser
from prance.util.resolver import RESOLVE_INTERNAL
from agno.agent import Agent
from agno.utils.pprint import pprint_run_response
from agno.models.google import Gemini
from agno.models.openrouter import OpenRouter
from IPython.display import display
from utils import extract_endpoints, load_prompts

### User Input

- OpenAPI specification path
- API endpoint for analysis
- Analysis type ( _summary_ / _security check_ / _edge cases detection_ )
- LLM Provider ( _google_ / _openrouter_ )
- LLM Model
- API Key

In [16]:
default_style = {'description_width': 'initial'}
spec_file_options = [
  "example-openapi.json",
  "example-openapi.yaml"
]
spec_file_input = widgets.Combobox(
  placeholder="File path",
  options=spec_file_options,
	description="OpenAPI specification (json/yaml): ",
  disabled=False,
  style=default_style
)
display(spec_file_input)

Combobox(value='', description='OpenAPI specification (json/yaml): ', options=('example-openapi.json', 'exampl…

In [17]:
# sanitize and extra file path
spec_file_path = spec_file_input.value.strip() or spec_file_options[0]
file_path = str(Path(spec_file_path).resolve())

# check for suppported file types
allowed_file_types = ['json', 'yaml', 'yml']
if file_path.split('.')[-1] not in allowed_file_types:
  raise ValueError('Only JSON or YAML file formats supported.')

# parse file
spec_parser = ResolvingParser(file_path, lazy=True, resolve_types=RESOLVE_INTERNAL)
spec = None
try:
  spec_parser.parse()
  spec = spec_parser.specification
except:
  raise Exception('Invalid OpenAPI specification. Failed to parse file') from None

# extract all endpoints
endpoints = extract_endpoints(spec)

# select endpoint
endpoint_select_options = [(f"{e['method'].upper()} {e['url']}", e) for e in endpoints]
endpoint_select = widgets.Dropdown(
  description="Endpoint: ",
  style=default_style,
  options=endpoint_select_options
)
# select analysis type
analysis_select_options = [
  ('Summary', 'summary'), 
  ('Security Vulnerabilities', 'security'), 
  ('Edge Cases detection', 'edge_cases')
]
analysis_select = widgets.Dropdown(
  description="Analysis type: ",
  style=default_style,
  options=analysis_select_options
)
# select model provider, model name, api key
model_provider_select = widgets.RadioButtons(
  description="Provider: ",
  style=default_style,
  options=[('Google', 'gemini'), ('OpenRouter', 'openrouter')]
)
model_name_input = widgets.Text(
  description="Model: ",
  style=default_style,
  placeholder="gemini-2.5-flash-lite",
)
api_key_input = widgets.Text(
  description="API Key: ",
  style=default_style,
  placeholder="AIz....",
)
display(endpoint_select)
display(analysis_select)
display(model_provider_select)
display(model_name_input)
display(api_key_input)

Dropdown(description='Endpoint: ', options=(('GET /users', {'method': 'get', 'url': '/users'}), ('POST /users'…

Dropdown(description='Analysis type: ', options=(('Summary', 'summary'), ('Security Vulnerabilities', 'securit…

RadioButtons(description='Provider: ', options=(('Google', 'gemini'), ('OpenRouter', 'openrouter')), style=Des…

Text(value='', description='Model: ', placeholder='gemini-2.5-flash-lite', style=TextStyle(description_width='…

Text(value='', description='API Key: ', placeholder='AIz....', style=TextStyle(description_width='initial'))

In [ ]:
endpoint = endpoint_select.value or endpoints[0]
analysis = analysis_select.value or analysis_select_options[0][1]
provider = model_provider_select.value
model_id = model_name_input.value or 'gemini-2.5-flash-lite'
api_key = api_key_input.value

if not api_key:
  raise ValueError('Please provide API Key to continue.') from None

### Agent initialization, prompt preparation, running the query

- Agno Agent initialization based on provider
- Prompt preparation for the analysis type
- Running the agent with the prompt and query

In [ ]:
endpoint_method, endpoint_url = endpoint.values()
endpoint_info = spec['paths'][endpoint_url][endpoint_method]
instructions = load_prompts(
	analysis=analysis,
	spec=endpoint_info
)
agent = None
if provider == 'gemini':
	agent = Agent(
		name="Endpoint Analysis Agent",
		model=Gemini(id=model_id, api_key=api_key),
		instructions=instructions,
		markdown=True
	)
else :
	agent = Agent(
		name="Endpoint Analysis Agent",
		model=OpenRouter(id=model_id, api_key=api_key),
		instructions=instructions,
		markdown=True
	)

# calling the agent with a prompt
try:
	response = agent.run(f"Perform {analysis} analysis with given instructions for this endpoint")
	pprint_run_response(response, markdown=True)
except:
	raise Exception('Failed to run the agent. Try again') from None